In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 1 — Install & HF Auth                     ║
# ╚══════════════════════════════════════════════════╝

!pip install -q \
    torch==2.10.0 \
    torchvision==0.25.0 \
    torchaudio==2.10.0 \
    --index-url https://download.pytorch.org/whl/cu128 \
    --force-reinstall --no-deps

!pip install -q \
    diffusers==0.31.0 \
    transformers==4.44.0 \
    accelerate \
    peft \
    xformers \
    huggingface_hub \
    Pillow \
    matplotlib
from kaggle_secrets import UserSecretsClient
import os, torch

try:
    token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    print("✅ HF_TOKEN loaded")
except Exception as e:
    raise RuntimeError(f"❌ HF_TOKEN not found in Add-ons > Secrets: {e}")

print(f"PyTorch {torch.__version__} | GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} — {p.total_memory // 1024**3} GB")

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 3 — Validate dataset & auto-caption       ║
# ╚══════════════════════════════════════════════════╝

import shutil
from pathlib import Path
from PIL import Image

!pip install -q gdown
import gdown

gdown.download("https://drive.google.com/uc?id=YOUR_FILE_ID", 
               "/kaggle/working/training_data.zip", quiet=False)
import zipfile
with zipfile.ZipFile("/kaggle/working/training_data.zip", "r") as z:
    z.extractall("/kaggle/working/training_data")

# SRC = "/kaggle/input/datasets/arezkicherfouh/training-data"
# SRC = "/kaggle/input/noobai-lora-v2/training_data"
SRC = "/kaggle/working/training_data"
TRIGGER = "fcs"
# ─────────────────────────────────────────────────────────────
DST = "/kaggle/working/training_data"
shutil.copytree(SRC, DST, dirs_exist_ok=True)

exts   = (".png", ".jpg", ".jpeg", ".webp")
images = [p for p in Path(DST).iterdir() if p.suffix.lower() in exts]

broken, auto_capped, sizes = [], [], []

for img_path in sorted(images):
    try:
        w, h = Image.open(img_path).size
        sizes.append((w, h))
    except Exception as e:
        broken.append(img_path.name); print(f"  ❌ Broken: {img_path.name} — {e}"); continue

    txt = img_path.with_suffix(".txt")
    if not txt.exists() or not txt.read_text().strip():
        auto_capped.append(img_path.name)
        txt.write_text(
            f"{TRIGGER}, son_goku, detailed_shading, high_quality, "
            "masterpiece, best_quality, absurdres")

valid = len(images) - len(broken)
print(f"{'─'*52}")
print(f"  ✅ Valid images     : {valid}")
print(f"  ❌ Broken (removed) : {len(broken)}")
print(f"  ⚠️  Auto-captioned  : {len(auto_capped)}")

if sizes:
    ws, hs = [s[0] for s in sizes], [s[1] for s in sizes]
    sq = sum(1 for w,h in sizes if w==h)
    print(f"  Square/non-square  : {sq}/{len(sizes)-sq} (non-square padded, nothing cropped)")
    print(f"  Width range        : {min(ws)}–{max(ws)} px")
    print(f"  Height range       : {min(hs)}–{max(hs)} px")

print(f"\n{'─'*52}")
if   valid < 50:   print("⚠️  Very few images — high overfit risk.")
elif valid < 100:  print("⚠️  Minimal — style will be loose.")
elif valid < 200:  print("✅ Good.")
elif valid <= 400: print("✅ Excellent — sweet spot for SDXL LoRA.")
else:              print("⚠️  Large — consider curating to your best 300.")

print("\n💡 Caption format (Danbooru tags — NoobAI's native language):")
print(f"   {TRIGGER}, son_goku, rope_bondage,")
print( "   cel_shading, masterpiece, best_quality, absurdres")

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 3b — WD14 Auto-Captioning                 ║
# ╚══════════════════════════════════════════════════╝
# Tags each image with Danbooru tags using WD14 tagger.
# Supports all content ratings (general, sensitive, explicit).
# Overwrites any existing .txt captions — run ONCE.

!pip install -q onnxruntime huggingface_hub Pillow numpy

import os, re
import numpy as np
from pathlib import Path
from PIL import Image
from huggingface_hub import hf_hub_download
import onnxruntime as rt

# ── Config ────────────────────────────────────────
DATA_DIR        = "/kaggle/working/training_data"
TRIGGER         = "fcs"
THRESHOLD       = 0.35        # lower = more tags, higher = fewer/more confident
CHAR_THRESHOLD  = 0.75        # character tag confidence (keep high)
OVERWRITE       = True        # set False to skip already-captioned images

# Tags to always append regardless of image content
QUALITY_SUFFIX  = "masterpiece, best_quality, absurdres"

# ── Download WD14 model (SmilingWolf/wd-v1-4-vit-tagger-v2) ──
print("Downloading WD14 tagger model...")
MODEL_PATH  = hf_hub_download("SmilingWolf/wd-v1-4-vit-tagger-v2", filename="model.onnx")
LABELS_PATH = hf_hub_download("SmilingWolf/wd-v1-4-vit-tagger-v2", filename="selected_tags.csv")
print("✅ Model downloaded")

# ── Load labels ───────────────────────────────────
import csv
with open(LABELS_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows   = list(reader)

tag_names   = [r["name"] for r in rows]
# category 0 = general, 4 = character, 9 = rating
rating_idx  = [i for i, r in enumerate(rows) if r["category"] == "9"]
general_idx = [i for i, r in enumerate(rows) if r["category"] == "0"]
char_idx    = [i for i, r in enumerate(rows) if r["category"] == "4"]

# ── Load ONNX model ───────────────────────────────
sess = rt.InferenceSession(MODEL_PATH, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
input_name  = sess.get_inputs()[0].name
target_size = sess.get_inputs()[0].shape[2]   # 448 for vit-v2
print(f"✅ Model loaded | input size: {target_size}×{target_size}")

# ── Preprocessing ─────────────────────────────────
def preprocess(img_path, size):
    img = Image.open(img_path).convert("RGBA")
    # Paste on white background (handles transparency)
    bg  = Image.new("RGBA", img.size, (255, 255, 255))
    bg.paste(img, mask=img.split()[3] if img.mode == "RGBA" else None)
    img = bg.convert("RGB")
    # Pad to square
    w, h   = img.size
    sq     = max(w, h)
    padded = Image.new("RGB", (sq, sq), (255, 255, 255))
    padded.paste(img, ((sq - w) // 2, (sq - h) // 2))
    padded = padded.resize((size, size), Image.BICUBIC)
    # BGR, float32
    arr = np.array(padded, dtype=np.float32)[:, :, ::-1]
    return arr[np.newaxis, :]

# ── Tag one image ─────────────────────────────────
def tag_image(img_path):
    arr    = preprocess(img_path, target_size)
    probs  = sess.run(None, {input_name: arr})[0][0]

    # Rating (pick highest)
    rating = {tag_names[i]: probs[i] for i in rating_idx}
    top_rating = max(rating, key=rating.get)

    # General tags above threshold
    general = [(tag_names[i], probs[i]) for i in general_idx if probs[i] >= THRESHOLD]
    general.sort(key=lambda x: -x[1])
    general_tags = [t for t, _ in general]

    # Character tags
    chars = [(tag_names[i], probs[i]) for i in char_idx if probs[i] >= CHAR_THRESHOLD]
    char_tags = [t for t, _ in chars]

    return general_tags, char_tags, top_rating

# ── Process all images ────────────────────────────
exts   = (".png", ".jpg", ".jpeg", ".webp")
images = sorted([p for p in Path(DATA_DIR).iterdir() if p.suffix.lower() in exts])

print(f"\nTagging {len(images)} images...")
skipped = 0

for i, img_path in enumerate(images):
    txt_path = img_path.with_suffix(".txt")
    if not OVERWRITE and txt_path.exists() and txt_path.read_text().strip():
        skipped += 1
        continue

    try:
        general_tags, char_tags, rating = tag_image(img_path)

        # Build caption: trigger first, then tags, then quality suffix
        all_tags = [TRIGGER] + char_tags + general_tags
        # Clean underscores → spaces (Danbooru style for NoobAI)
        all_tags = [t.replace("_", " ") for t in all_tags]
        caption  = ", ".join(all_tags) + ", " + QUALITY_SUFFIX

        txt_path.write_text(caption, encoding="utf-8")

        if (i + 1) % 20 == 0 or (i + 1) == len(images):
            print(f"  [{i+1}/{len(images)}] {img_path.name}")
            print(f"    rating: {rating} | tags: {len(general_tags)} general, {len(char_tags)} char")
            print(f"    → {caption[:120]}...")

    except Exception as e:
        print(f"  ❌ Failed: {img_path.name} — {e}")

print(f"\n✅ Done — {len(images) - skipped} captioned, {skipped} skipped")
print("Run Cell 3 again to validate the dataset.")

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/training_data_captioned", "zip", "/kaggle/working/training_data")
print("✅ /kaggle/working/training_data_captioned.zip ready to download")

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 2 — Training script                       ║
# ╚══════════════════════════════════════════════════╝
%%writefile train_anime_lora.py
import os, gc, torch, math
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import CLIPTokenizer, CLIPTextModel, CLIPTextModelWithProjection
from diffusers import UNet2DConditionModel, AutoencoderKL, DDPMScheduler
from peft import LoraConfig, get_peft_model
from huggingface_hub import HfApi
from accelerate import Accelerator
from accelerate.utils import set_seed
from kaggle_secrets import UserSecretsClient
import torch.nn.functional as F

# ── Config ────────────────────────────────────────────────────
HF_TOKEN   = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

BASE_MODEL = "Laxhar/noobai-XL-Vpred-1.0"
REPO_ID    = "Arezki-Cherfouh/noobai-xl-lora"
DATA_DIR   = "/kaggle/working/training_data"
OUTPUT_DIR = "/kaggle/working/lora_output"
TRIGGER    = "fcs"
RESOLUTION = 768        # ↓ from 1024 — saves ~40% activation memory
BATCH_SIZE = 1
EPOCHS     = 1
GRAD_ACCUM = 4          # ↑ from 2 — compensates for smaller resolution
LR         = 1e-4
LORA_RANK  = 8          # ↓ from 16 — lighter memory footprint
LORA_ALPHA = 32         # keep ratio: alpha = rank * 4
LORA_DROP  = 0.1
MAX_GRAD   = 1.0
SEED       = 42

# RESOLUTION = 1024    # back to original
# GRAD_ACCUM = 8       # back to original  
# LORA_RANK  = 32      # back to original
# LORA_ALPHA = 64      # back to original
# EPOCHS     = 3       # back to original

set_seed(SEED)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
api = HfApi(token=HF_TOKEN)
print(f"GPUs: {torch.cuda.device_count()} | BASE: {BASE_MODEL}")

# ── Smart resize — pad to square, no crop, any input res ──────
def pad_to_square(img, fill=(255, 255, 255)):
    w, h = img.size
    size = max(w, h)
    out  = Image.new("RGB", (size, size), fill)
    out.paste(img, ((size - w) // 2, (size - h) // 2))
    return out

def smart_resize(img, target):
    img = img.convert("RGB")
    if img.size[0] != img.size[1]:
        img = pad_to_square(img)
    if img.size[0] != target:
        img = img.resize((target, target), Image.LANCZOS)
    return img

# ── Dataset ───────────────────────────────────────────────────
class AnimeDataset(Dataset):
    def __init__(self, data_dir, tok1, tok2, resolution=768):
        self.root = Path(data_dir)
        self.imgs = []
        for ext in ("*.png", "*.jpg", "*.jpeg", "*.webp"):
            self.imgs += sorted(self.root.glob(ext))
        self.tok1, self.tok2, self.res = tok1, tok2, resolution
        self.to_tensor = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])
        print(f"Dataset: {len(self.imgs)} images → padded to {resolution}×{resolution}")

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        img_path = self.imgs[idx]
        txt_path = img_path.with_suffix(".txt")
        caption  = txt_path.read_text(encoding="utf-8").strip() if txt_path.exists() else ""
        if not caption:
            caption = f"{TRIGGER}, 1girl, detailed_shading, masterpiece, best_quality, absurdres"
        elif TRIGGER not in caption:
            caption = f"{TRIGGER}, {caption}"

        pv = self.to_tensor(smart_resize(Image.open(img_path), self.res))

        t1 = self.tok1(caption, truncation=True, max_length=77, return_tensors="pt", padding="max_length")
        t2 = self.tok2(caption, truncation=True, max_length=77, return_tensors="pt", padding="max_length")
        return {"pixel_values": pv,
                "input_ids_1":  t1.input_ids.squeeze(0),
                "input_ids_2":  t2.input_ids.squeeze(0)}

# ── Load components ───────────────────────────────────────────
print("Loading tokenizers...")
tok1 = CLIPTokenizer.from_pretrained(BASE_MODEL, subfolder="tokenizer",   token=HF_TOKEN)
tok2 = CLIPTokenizer.from_pretrained(BASE_MODEL, subfolder="tokenizer_2", token=HF_TOKEN)

print("Loading text encoders (frozen)...")
enc1 = CLIPTextModel.from_pretrained(
    BASE_MODEL, subfolder="text_encoder", token=HF_TOKEN, torch_dtype=torch.float16)
enc2 = CLIPTextModelWithProjection.from_pretrained(
    BASE_MODEL, subfolder="text_encoder_2", token=HF_TOKEN, torch_dtype=torch.float16)

print("Loading VAE (frozen)...")
vae = AutoencoderKL.from_pretrained(
    BASE_MODEL, subfolder="vae", token=HF_TOKEN, torch_dtype=torch.float16)

print("Loading scheduler (v-prediction + zero SNR)...")
sched = DDPMScheduler.from_pretrained(BASE_MODEL, subfolder="scheduler", token=HF_TOKEN)
sched.config.prediction_type        = "v_prediction"
sched.config.rescale_betas_zero_snr = True
print(f"  prediction_type        : {sched.config.prediction_type}")
print(f"  rescale_betas_zero_snr : {sched.config.rescale_betas_zero_snr}")

print("Loading UNet (fp16 + gradient checkpointing)...")
unet = UNet2DConditionModel.from_pretrained(
    BASE_MODEL, subfolder="unet", token=HF_TOKEN,
    torch_dtype=torch.float16)   # fp16 — saves ~6 GB vs float32

# Freeze everything — only LoRA adapters will train
for m in [enc1, enc2, vae, unet]:
    m.requires_grad_(False)

lora_cfg = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROP,
    bias="none",
    target_modules=["to_q","to_k","to_v","to_out.0",
                    "proj_in","proj_out",
                    "ff.net.0.proj","ff.net.2",
                    "add_k_proj","add_v_proj"],
    init_lora_weights="gaussian",
)
unet = get_peft_model(unet, lora_cfg)
unet.enable_gradient_checkpointing()   # trades compute for memory
unet.print_trainable_parameters()

# ── Accelerator ───────────────────────────────────────────────
acc    = Accelerator(gradient_accumulation_steps=GRAD_ACCUM, mixed_precision="fp16")
device = acc.device

gc.collect()
torch.cuda.empty_cache()

enc1.to(device); enc2.to(device); vae.to(device)

optim = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, unet.parameters()),
    lr=LR, weight_decay=1e-2, eps=1e-8)

ds = AnimeDataset(DATA_DIR, tok1, tok2, RESOLUTION)
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)

unet, optim, dl = acc.prepare(unet, optim, dl)

lr_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim, T_max=EPOCHS * math.ceil(len(dl) / GRAD_ACCUM), eta_min=1e-6)

# ── Training loop ─────────────────────────────────────────────
print(f"\n🎨 {EPOCHS} epochs | {len(dl)} steps/ep | "
      f"eff. batch = {BATCH_SIZE * GRAD_ACCUM * acc.num_processes}")
global_step = 0

for epoch in range(EPOCHS):
    unet.train()
    running = 0.0
    for step, batch in enumerate(dl):
        with acc.accumulate(unet):

            with torch.no_grad():
                latents = vae.encode(
                    batch["pixel_values"].to(device, dtype=torch.float16)
                ).latent_dist.sample() * vae.config.scaling_factor

            noise     = torch.randn_like(latents)
            bsz       = latents.shape[0]
            timesteps = torch.randint(
                0, sched.config.num_train_timesteps, (bsz,), device=device).long()
            noisy = sched.add_noise(latents, noise, timesteps)

            with torch.no_grad():
                e1 = enc1(batch["input_ids_1"].to(device), output_hidden_states=True)
                e2 = enc2(batch["input_ids_2"].to(device), output_hidden_states=True)
                embeds = torch.cat([e1.hidden_states[-2], e2.hidden_states[-2]], dim=-1)
                pooled = e2[0]

            add_ids = torch.tensor(
                [[RESOLUTION, RESOLUTION, 0, 0, RESOLUTION, RESOLUTION]] * bsz,
                dtype=torch.float16, device=device)
            added = {"text_embeds": pooled.half(), "time_ids": add_ids}

            pred = unet(
                noisy, timesteps,
                encoder_hidden_states=embeds.half(),
                added_cond_kwargs=added,
            ).sample

            target = sched.get_velocity(
                latents.float(), noise.float(), timesteps)

            loss = F.mse_loss(pred.float(), target.float(), reduction="mean")
            running += loss.item()

            acc.backward(loss)
            if acc.sync_gradients:
                acc.clip_grad_norm_(unet.parameters(), MAX_GRAD)
            optim.step()
            lr_sched.step()
            optim.zero_grad()

        global_step += 1
        if global_step % 20 == 0 and acc.is_main_process:
            print(f"  [Ep {epoch+1}/{EPOCHS}] step {global_step} "
                  f"loss={running/(step+1):.5f} "
                  f"lr={optim.param_groups[0]['lr']:.2e}")

    if acc.is_main_process:
        print(f"\n✅ Epoch {epoch+1} | avg loss {running/len(dl):.5f}\n")

# ── Save ──────────────────────────────────────────────────────
acc.wait_for_everyone()
if acc.is_main_process:
    print("Saving LoRA adapter...")
    unwrapped = acc.unwrap_model(unet)
    unwrapped.save_pretrained(OUTPUT_DIR)
    tok1.save_pretrained(OUTPUT_DIR)
    tok2.save_pretrained(OUTPUT_DIR)
    sched.save_pretrained(OUTPUT_DIR + "/scheduler")
    print(f"✅ Saved to {OUTPUT_DIR}")
    print("Run Cell 5 to test inference, Cell 6 to push.")

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 4 — Launch dual-T4 training               ║
# ╚══════════════════════════════════════════════════╝
#  ~2–4 hrs depending on dataset size

!PYTORCH_ALLOC_CONF=expandable_segments:True \
 accelerate launch \
   --num_processes=2 \
   --multi_gpu \
   --mixed_precision=fp16 \
   --dynamo_backend=no \
 train_anime_lora.py

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 5 — Inference test: 2×2 image grid        ║
# ╚══════════════════════════════════════════════════╝
#
# Loads across BOTH T4s using device_map="balanced"
# to avoid OOM from fitting full SDXL pipeline on one GPU.

import torch, gc, os
from pathlib import Path
from diffusers import StableDiffusionXLPipeline, EulerDiscreteScheduler
from peft import PeftModel
from transformers import CLIPTextModel, CLIPTextModelWithProjection
from kaggle_secrets import UserSecretsClient
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

HF_TOKEN   = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

BASE_MODEL = "Laxhar/noobai-XL-Vpred-1.0"
LORA_DIR   = "/kaggle/working/lora_output"
OUT_DIR    = Path("/kaggle/working/test_outputs")
OUT_DIR.mkdir(exist_ok=True)

gc.collect()
torch.cuda.empty_cache()

# ── Load pipeline spread across both T4s ─────────────────────
# device_map="balanced" splits UNet layers across GPU 0 + GPU 1
# so neither card OOMs on the ~6GB UNet alone.
print("Loading pipeline (balanced across both T4s)...")
pipe = StableDiffusionXLPipeline.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    token=HF_TOKEN,
    use_safetensors=True,
    device_map="balanced",         # splits across cuda:0 + cuda:1
    max_memory={0: "14GiB", 1: "14GiB"},
)

# ── V-prediction scheduler — REQUIRED for NoobAI-XL ──────────
# Wrong scheduler (e.g. default DDPM or DDIM without v-pred)
# produces grey, washed-out, or pure noise output.
pipe.scheduler = EulerDiscreteScheduler.from_config(
    pipe.scheduler.config,
    prediction_type="v_prediction",
    rescale_betas_zero_snr=True,
    timestep_spacing="trailing",
)
print(f"✅ Scheduler: EulerDiscrete | v_prediction | zero SNR | trailing")

# ── Memory optimizations for VAE decode ──────────────────────
# VAE decode at 1024×1024 spikes ~1 GB — tiling breaks it into
# smaller spatial chunks so neither T4 runs out of memory.
# Remove both lines if you upgrade to A100/H100 with more VRAM.
pipe.enable_vae_slicing()   # decodes one image at a time (safe default)
pipe.enable_vae_tiling()    # decodes in tiles — fixes the 1 GB OOM spike

# ── Inject LoRA weights via PEFT ──────────────────────────────
# pipe.unet.load_adapter() does NOT exist in diffusers 0.27.2
# The adapter was saved with peft's save_pretrained so load it the same way.
pipe.unet = PeftModel.from_pretrained(
    pipe.unet,
    LORA_DIR,
    is_trainable=False,
)
pipe.unet.eval()
print("✅ LoRA weights injected via PEFT")

# ── Test prompts — Danbooru tags ──────────────────────────────
TEST_PROMPTS = [
    (
        "fcs, son_goku, blue_qipao, rope_bondage, sitting, wooden_chair, "
        "sharp_linework, cel_shading, "
        "high_contrast_shadows, masterpiece, best_quality, absurdres",
        "Test 1 — Style reference match"
    ),
    (
        "fcs, son_goku, white_uniform, standing, portrait, "
        "detailed_eyes, soft_lighting, "
        "masterpiece, best_quality, absurdres",
        "Test 2 — Portrait"
    ),
    (
        "fcs, son_goku, dynamic_pose, full_body, wind, "
        "detailed_outfit, dramatic_lighting, "
        "masterpiece, best_quality, absurdres",
        "Test 3 — Action / full body"
    ),
    (
        "fcs, son_goku, sitting, crossed_legs, "
        ""
        "elegant_interior, masterpiece, best_quality, absurdres",
        "Test 4 — Action pose"
    ),
]
NEGATIVE = (
    "lowres, bad_anatomy, bad_hands, missing_fingers, worst_quality, "
    "low_quality, jpeg_artifacts, blurry, 3d, photo, realistic, "
    ""
)

# ── Generate ──────────────────────────────────────────────────
# width/height: 768 for dual-T4 smoke test — change back to 1024
# once you confirm the pipeline works end-to-end, or on better hardware.
# Original: width=1024, height=1024
results = []
for i, (prompt, label) in enumerate(TEST_PROMPTS):
    print(f"  [{i+1}/4] {label}...")
    with torch.no_grad():
        img = pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE,
            num_inference_steps=28,
            guidance_scale=7.0,
            width=768, height=768,   # ← 1024 on A100 / after smoke test passes
            generator=torch.Generator("cpu").manual_seed(i * 1000 + 42),
        ).images[0]
    img.save(OUT_DIR / f"test_{i+1}.png")
    results.append((img, label))
    print(f"     ✅ saved")

# ── 2×2 display grid ──────────────────────────────────────────
fig = plt.figure(figsize=(18, 18))
gs  = gridspec.GridSpec(2, 2, figure=fig, wspace=0.05, hspace=0.1)
for idx, (img, label) in enumerate(results):
    ax = fig.add_subplot(gs[idx//2, idx%2])
    ax.imshow(img); ax.set_title(label, fontsize=11, pad=6); ax.axis("off")
fig.suptitle("NoobAI-XL LoRA Test — fcs trigger", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / "test_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nTroubleshooting:")
print("  Grey/washed output → scheduler v-pred config error")
print("  Weak style         → increase LORA_RANK=64 or EPOCHS=4 in Cell 2")
print("  Bad anatomy        → add more varied pose images to dataset")



# # ╔══════════════════════════════════════════════════╗
# # ║  CELL 5 — Inference test: 2×2 image grid        ║
# # ╚══════════════════════════════════════════════════╝
# #
# # Loads across BOTH T4s using device_map="balanced"
# # to avoid OOM from fitting full SDXL pipeline on one GPU.

# import torch, gc, os
# from pathlib import Path
# from diffusers import StableDiffusionXLPipeline, EulerDiscreteScheduler
# from peft import PeftModel
# from transformers import CLIPTextModel, CLIPTextModelWithProjection
# from kaggle_secrets import UserSecretsClient
# import matplotlib.pyplot as plt
# import matplotlib.gridspec as gridspec

# HF_TOKEN   = UserSecretsClient().get_secret("HF_TOKEN")
# os.environ["HF_TOKEN"] = HF_TOKEN

# BASE_MODEL = "Laxhar/noobai-XL-Vpred-1.0"
# LORA_DIR   = "/kaggle/working/lora_output"
# OUT_DIR    = Path("/kaggle/working/test_outputs")
# OUT_DIR.mkdir(exist_ok=True)

# gc.collect()
# torch.cuda.empty_cache()

# # ── Load pipeline spread across both T4s ─────────────────────
# # device_map="balanced" splits UNet layers across GPU 0 + GPU 1
# # so neither card OOMs on the ~6GB UNet alone.
# print("Loading pipeline (balanced across both T4s)...")
# pipe = StableDiffusionXLPipeline.from_pretrained(
#     BASE_MODEL,
#     torch_dtype=torch.float16,
#     token=HF_TOKEN,
#     use_safetensors=True,
#     device_map="balanced",         # splits across cuda:0 + cuda:1
#     max_memory={0: "14GiB", 1: "14GiB"},
# )

# # ── V-prediction scheduler — REQUIRED for NoobAI-XL ──────────
# # Wrong scheduler (e.g. default DDPM or DDIM without v-pred)
# # produces grey, washed-out, or pure noise output.
# pipe.scheduler = EulerDiscreteScheduler.from_config(
#     pipe.scheduler.config,
#     prediction_type="v_prediction",
#     rescale_betas_zero_snr=True,
#     timestep_spacing="trailing",
# )
# print(f"✅ Scheduler: EulerDiscrete | v_prediction | zero SNR | trailing")

# # ── Inject LoRA weights via PEFT ──────────────────────────────
# # pipe.unet.load_adapter() does NOT exist in diffusers 0.27.2
# # The adapter was saved with peft's save_pretrained so load it the same way.
# from peft import PeftModel
# pipe.unet = PeftModel.from_pretrained(
#     pipe.unet,
#     LORA_DIR,
#     is_trainable=False,
# )
# pipe.unet.eval()
# print("✅ LoRA weights injected via PEFT")

# # ── Test prompts — Danbooru tags ──────────────────────────────
# TEST_PROMPTS = [
#     (
#         "fcs, son_goku, blue_qipao, rope_bondage, sitting, wooden_chair, "
#         "sharp_linework, cel_shading, "
#         "high_contrast_shadows, masterpiece, best_quality, absurdres",
#         "Test 1 — Style reference match"
#     ),
#     (
#         "fcs, son_goku, white_uniform, standing, portrait, "
#         "detailed_eyes, soft_lighting, "
#         "masterpiece, best_quality, absurdres",
#         "Test 2 — Portrait"
#     ),
#     (
#         "fcs, son_goku, dynamic_pose, full_body, wind, "
#         "detailed_outfit, dramatic_lighting, "
#         "masterpiece, best_quality, absurdres",
#         "Test 3 — Action / full body"
#     ),
#     (
#         "fcs, son_goku, sitting, crossed_legs, "
#         ""
#         "elegant_interior, masterpiece, best_quality, absurdres",
#         "Test 4 — Action pose"
#     ),
# ]
# NEGATIVE = (
#     "lowres, bad_anatomy, bad_hands, missing_fingers, worst_quality, "
#     "low_quality, jpeg_artifacts, blurry, 3d, photo, realistic, "
#     ""
# )

# # ── Generate ──────────────────────────────────────────────────
# results = []
# for i, (prompt, label) in enumerate(TEST_PROMPTS):
#     print(f"  [{i+1}/4] {label}...")
#     with torch.no_grad():
#         img = pipe(
#             prompt=prompt,
#             negative_prompt=NEGATIVE,
#             num_inference_steps=28,
#             guidance_scale=7.0,
#             width=1024, height=1024,
#             generator=torch.Generator("cpu").manual_seed(i * 1000 + 42),
#         ).images[0]
#     img.save(OUT_DIR / f"test_{i+1}.png")
#     results.append((img, label))
#     print(f"     ✅ saved")

# # ── 2×2 display grid ──────────────────────────────────────────
# fig = plt.figure(figsize=(18, 18))
# gs  = gridspec.GridSpec(2, 2, figure=fig, wspace=0.05, hspace=0.1)
# for idx, (img, label) in enumerate(results):
#     ax = fig.add_subplot(gs[idx//2, idx%2])
#     ax.imshow(img); ax.set_title(label, fontsize=11, pad=6); ax.axis("off")
# fig.suptitle("NoobAI-XL LoRA Test — fcs trigger", fontsize=14, fontweight="bold", y=1.01)
# plt.tight_layout()
# plt.savefig(OUT_DIR / "test_grid.png", dpi=150, bbox_inches="tight")
# plt.show()
# print("\nTroubleshooting:")
# print("  Grey/washed output → scheduler v-pred config error")
# print("  Weak style         → increase LORA_RANK=64 or EPOCHS=4 in Cell 2")
# print("  Bad anatomy        → add more varied pose images to dataset")

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 6 — Push adapter + model card to Hub      ║
# ╚══════════════════════════════════════════════════╝
#
# Pushes only the LoRA adapter (small — a few hundred MB),
# NOT the full 6GB base model. Users load base + adapter separately.

from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
from pathlib import Path
import json, os

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
REPO_ID  = "Arezki-Cherfouh/noobai-xl-lora"   # ← CHANGE (match Cell 2)
LORA_DIR = "/kaggle/working/lora_output"

api = HfApi(token=HF_TOKEN)

# ── Write adapter_config.json if missing ─────────────────────
# Without this, HF Hub can't identify the repo as a valid adapter.
adapter_cfg_path = Path(LORA_DIR) / "adapter_config.json"
if not adapter_cfg_path.exists():
    adapter_cfg = {
        "base_model_name_or_path": "Laxhar/noobai-XL-Vpred-1.",
        "peft_type": "LORA",
        "task_type": "FEATURE_EXTRACTION",
        "inference_mode": True,
        "r": 32,
        "lora_alpha": 64,
        "lora_dropout": 0.1,
        "bias": "none",
    }
    adapter_cfg_path.write_text(json.dumps(adapter_cfg, indent=2))
    print("✅ adapter_config.json written")

# ── Model card ────────────────────────────────────────────────
card = """---
license: other
base_model: Laxhar/noobai-XL-Vpred-1.0
tags:
  - lora
  - sdxl
  - illustrious
  - noobai
  - anime
  - text-to-image
  - diffusers
  - peft
  - v-prediction
pipeline_tag: text-to-image
---

# Anime LoRA — Custom Style (NoobAI-XL / Illustrious)

LoRA adapter for [Laxhar/noobai-XL-Vpred-1.0](https://huggingface.co/Laxhar/noobai-XL-Vpred-1.0).

> ⚠️ **V-prediction model.** Use `EulerDiscreteScheduler` with `prediction_type="v_prediction"`,
> `rescale_betas_zero_snr=True`, `timestep_spacing="trailing"`. Wrong scheduler = grey output.

**Trigger token:** `fcs` | **Tag style:** Danbooru

## Quick start

```python
from diffusers import StableDiffusionXLPipeline, EulerDiscreteScheduler
from peft import PeftModel
import torch

pipe = StableDiffusionXLPipeline.from_pretrained(
    "Laxhar/noobai-XL-Vpred-1.0",
    torch_dtype=torch.float16,
    device_map="balanced",
)
pipe.scheduler = EulerDiscreteScheduler.from_config(
    pipe.scheduler.config,
    prediction_type="v_prediction",
    rescale_betas_zero_snr=True,
    timestep_spacing="trailing",
)
pipe.unet = PeftModel.from_pretrained(
    pipe.unet, "YOUR_HF_USERNAME/noobai-xl-lora", is_trainable=False
)
pipe.unet.eval()

image = pipe(
    "fcs, son_goku, detailed_shading, masterpiece, best_quality, absurdres",
    negative_prompt="lowres, bad_anatomy, worst_quality",
    num_inference_steps=28, guidance_scale=7.0,
    width=1024, height=1024,
).images[0]
```

## Training details

| Setting | Value |
|---|---|
| Base | Laxhar/noobai-XL-Vpred-1.0 (Illustrious-XL) |
| Scheduler | v-prediction + zero terminal SNR |
| LoRA rank / alpha | 32 / 64 |
| Resolution | 1024×1024 (pad to square, no crop) |
| Epochs | 3 |
| Hardware | 2× NVIDIA T4 16GB (Kaggle) |
| Trigger | `fcs` |
| Tag style | Danbooru |
"""
Path(LORA_DIR + "/README.md").write_text(card)
print("✅ Model card written")

# ── Create repo and push ──────────────────────────────────────
api.create_repo(repo_id=REPO_ID, repo_type="model", exist_ok=True, private=True)
api.upload_folder(
    folder_path=LORA_DIR,
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Add NoobAI-XL LoRA adapter",
)
print(f"\n✅ https://huggingface.co/{REPO_ID}")
print("Load with: PeftModel.from_pretrained(pipe.unet, REPO_ID)")

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 7 — Merge LoRA into base & push (optional)║
# ╚══════════════════════════════════════════════════╝

from peft import PeftModel
from diffusers import StableDiffusionXLPipeline
import torch, os
from kaggle_secrets import UserSecretsClient

HF_TOKEN   = UserSecretsClient().get_secret("HF_TOKEN")
MERGED_DIR = "/kaggle/working/merged_model"
REPO_ID    = "Arezki-Cherfouh/noobai-xl-lora-merged"  # separate repo

pipe = StableDiffusionXLPipeline.from_pretrained(
    "Laxhar/noobai-XL-Vpred-1.0",
    torch_dtype=torch.float16,
    token=HF_TOKEN,
)

pipe.unet = PeftModel.from_pretrained(pipe.unet, "/kaggle/working/lora_output")
pipe.unet = pipe.unet.merge_and_unload()  # fuses LoRA weights into base UNet

pipe.save_pretrained(MERGED_DIR)
pipe.push_to_hub(REPO_ID, token=HF_TOKEN, private=True)
print(f"✅ Merged model pushed to {REPO_ID}")

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 8 — Load from Hub & generate images       ║
# ╚══════════════════════════════════════════════════╝

import torch, gc, os
from pathlib import Path
from diffusers import StableDiffusionXLPipeline, EulerDiscreteScheduler
from peft import PeftModel
from kaggle_secrets import UserSecretsClient
import matplotlib.pyplot as plt

HF_TOKEN   = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

# ╔══════════════════════════════════════════════════╗
# ║  USER CONFIG — edit only this block             ║
# ╚══════════════════════════════════════════════════╝

# ── Loading mode ──────────────────────────────────
# "adapter"  → loads base model + LoRA adapter from Hub (lightweight, recommended)
# "merged"   → loads fully merged model from Hub (standalone, no base needed)
LOAD_MODE   = "adapter"

BASE_MODEL  = "Laxhar/noobai-XL-Vpred-1.0"            # ignored if LOAD_MODE="merged"
ADAPTER_ID  = "Arezki-Cherfouh/noobai-xl-lora"        # used if LOAD_MODE="adapter"
MERGED_ID   = "Arezki-Cherfouh/noobai-xl-lora-merged" # used if LOAD_MODE="merged"

# ── Generation settings ───────────────────────────
NUM_IMAGES  = 4        # how many images to generate
WIDTH       = 768      # ← 1024 if VRAM allows
HEIGHT      = 768      # ← 1024 if VRAM allows
STEPS       = 28
CFG         = 7.0
SEED        = 42       # ← set to None for random seeds

PROMPT = (
    "fcs, son_goku, masterpiece, best_quality, absurdres"
    # ↑ edit to whatever you want to generate
)
NEGATIVE = (
    "lowres, bad_anatomy, bad_hands, missing_fingers, worst_quality, "
    "low_quality, jpeg_artifacts, blurry, 3d, photo, realistic"
)

OUT_DIR = Path("/kaggle/working/custom_outputs")
# ╚══════════════════════════════════════════════════╝

OUT_DIR.mkdir(exist_ok=True)
gc.collect()
torch.cuda.empty_cache()

# ── Load pipeline ─────────────────────────────────────────────
if LOAD_MODE == "adapter":
    print(f"Loading base model + adapter ({ADAPTER_ID})...")
    pipe = StableDiffusionXLPipeline.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        token=HF_TOKEN,
        use_safetensors=True,
        device_map="balanced",
        max_memory={0: "14GiB", 1: "14GiB"},
    )
    pipe.unet = PeftModel.from_pretrained(
        pipe.unet,
        ADAPTER_ID,
        token=HF_TOKEN,
        is_trainable=False,
    )
    pipe.unet.eval()
    print(f"✅ Base + adapter loaded")

elif LOAD_MODE == "merged":
    print(f"Loading merged model ({MERGED_ID})...")
    pipe = StableDiffusionXLPipeline.from_pretrained(
        MERGED_ID,
        torch_dtype=torch.float16,
        token=HF_TOKEN,
        use_safetensors=True,
        device_map="balanced",
        max_memory={0: "14GiB", 1: "14GiB"},
    )
    print(f"✅ Merged model loaded")

else:
    raise ValueError(f'LOAD_MODE must be "adapter" or "merged", got "{LOAD_MODE}"')

# ── Scheduler (required for both modes) ──────────────────────
pipe.scheduler = EulerDiscreteScheduler.from_config(
    pipe.scheduler.config,
    prediction_type="v_prediction",
    rescale_betas_zero_snr=True,
    timestep_spacing="trailing",
)
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()
print("✅ Scheduler: EulerDiscrete | v_prediction | zero SNR | trailing")

# ── Generate ──────────────────────────────────────────────────
print(f"\nGenerating {NUM_IMAGES} image(s)...")
results = []
for i in range(NUM_IMAGES):
    seed = (SEED + i) if SEED is not None else torch.randint(0, 2**32, (1,)).item()
    print(f"  [{i+1}/{NUM_IMAGES}] seed={seed}")
    with torch.no_grad():
        img = pipe(
            prompt=PROMPT,
            negative_prompt=NEGATIVE,
            num_inference_steps=STEPS,
            guidance_scale=CFG,
            width=WIDTH,
            height=HEIGHT,
            generator=torch.Generator("cpu").manual_seed(seed),
        ).images[0]
    img.save(OUT_DIR / f"output_{i+1}_seed{seed}.png")
    results.append((img, f"seed {seed}"))
    print(f"     ✅ saved")

# ── Display grid ──────────────────────────────────────────────
cols = min(NUM_IMAGES, 4)
rows = (NUM_IMAGES + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
axes = [axes] if NUM_IMAGES == 1 else (
    [ax for row in axes for ax in row] if rows > 1 else list(axes)
)
for idx, (img, label) in enumerate(results):
    axes[idx].imshow(img)
    axes[idx].set_title(label, fontsize=10)
    axes[idx].axis("off")
for idx in range(NUM_IMAGES, len(axes)):
    axes[idx].axis("off")

plt.suptitle(f"NoobAI-XL — {ADAPTER_ID if LOAD_MODE == 'adapter' else MERGED_ID}",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "output_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\n✅ All images saved to {OUT_DIR}")